# High-Speed Overture Maps & OpenStreetMap Buildings Extractor

This notebook queries buildings in a specified bounding box using **parallelized cloud partition pruning** (via the STAC-geoparquet index) and **vectorized geometry parsing** (using Shapely 2.0 C-bindings). It outputs a vector CSV file and a raster GeoTIFF file (`buildings.csv` and `buildings.tif`) for QGIS visualization in seconds.

## Freshness & Ghost Building Cleanup:
1. **STAC-based Cloud Pruning**: Queries Overture Live and OSM Live to get the most up-to-date compiled raw footprints.
2. **Automated 2026 Satellite Verification**: Downloads the latest cloud-free **2026 Sentinel-2 satellite imagery** (10m resolution) using the Microsoft Planetary Computer API.
3. **Vegetation Index Filtering (NDVI)**: Calculates the vegetation index (NDVI) inside each building footprint. If a building footprint in 2026 is covered by high vegetation (NDVI > 0.35), indicating it was demolished, cleared, or overgrown, the script automatically flags it as a **ghost building** and removes it from the final output.

## How to use:
1. Modify the coordinates in **Section 1: Configuration** below.
2. Toggle `enable_ghost_cleanup` to `True` or `False`.
3. Run all cells in this notebook.
4. Open the output CSV and GeoTIFF files directly in QGIS.

### Section 1: Configuration

In [ ]:
# --- TARGET CONFIGURATION ---
# Bounding Box Coordinates (WGS84 Lat/Lon)
xmin = -69.8040  # Min Longitude (West)
ymin = 9.9660    # Min Latitude (South)
xmax = -69.7950  # Max Longitude (East)
ymax = 9.9750    # Max Latitude (North)

# Output File Paths
csv_out = "buildings.csv"
tif_out = "buildings.tif"

# Output Settings
resolution = 0.00001     # Raster resolution in degrees per pixel (default: 0.00001, ~1.1 meters)

# Data Source Selection:
# - "live"          : (DEFAULT) Combines both Overture Live and OpenStreetMap Live, keeping the freshest updates and removing duplicates
# - "overture_live"  : Queries the latest live Overture Maps dataset from the cloud using STAC parallel pruning (June 2026 Release)
# - "osm_live"       : Queries real-time OpenStreetMap data directly using the Overpass API (mapped up to today)
data_source = "live"

# Ghost Building Cleanup:
# - Set to True to automatically fetch latest 2026 Sentinel-2 satellite imagery and drop buildings with vegetated/demolished signatures
# - Set to False to keep all raw footprints without satellite verification
enable_ghost_cleanup = True

### Section 2: Imports & Helper Functions

In [ ]:
import sys
import os
import ast
import json
import duckdb
import urllib.request
import urllib.parse
import pandas as pd
import numpy as np
import rasterio
import shapely
from datetime import date
from rasterio.transform import from_bounds
from rasterio.features import rasterize, geometry_mask
from rasterio.warp import transform_bounds
from rasterio.windows import from_bounds as win_from_bounds
import overturemaps

def get_max_update_time(sources_val):
    if not sources_val:
        return None
    try:
        if isinstance(sources_val, str):
            src_list = ast.literal_eval(sources_val)
        elif isinstance(sources_val, list):
            src_list = sources_val
        else:
            return None
        
        times = []
        for src in src_list:
            if isinstance(src, dict):
                ut = src.get('update_time')
                if ut:
                    times.append(ut[:10])
        if times:
            return max(times)
    except Exception:
        pass
    return None

def fetch_osm_live(xmin, ymin, xmax, ymax):
    print("Querying real-time OpenStreetMap data via Overpass API...")
    overpass_url = 'https://overpass-api.de/api/interpreter'
    overpass_query = f"""
    [out:json][timeout:30];
    (
      node["building"]({ymin},{xmin},{ymax},{xmax});
      way["building"]({ymin},{xmin},{ymax},{xmax});
      relation["building"]({ymin},{xmin},{ymax},{xmax});
    );
    out body;
    >;
    out skel qt;
    """
    
    data = urllib.parse.urlencode({'data': overpass_query}).encode('utf-8')
    req = urllib.request.Request(overpass_url, data=data, headers={'User-Agent': 'BuildingExporter/2.0'})
    
    today_str = date.today().isoformat()
    try:
        with urllib.request.urlopen(req) as response:
            res_json = json.loads(response.read().decode('utf-8'))
            elements = res_json.get('elements', [])
            
            # Index nodes by ID
            nodes = {el['id']: (el['lon'], el['lat']) for el in elements if el['type'] == 'node'}
            ways = [el for el in elements if el['type'] == 'way']
            
            osm_buildings = []
            for way in ways:
                way_nodes = way.get('nodes', [])
                coords = [nodes[node_id] for node_id in way_nodes if node_id in nodes]
                if len(coords) >= 4 and coords[0] == coords[-1]:
                    tags = way.get('tags', {})
                    height_val = tags.get('height', tags.get('building:levels', '3.0'))
                    try:
                        height = float(height_val) * 3.0 if ':' in height_val else float(height_val)
                    except ValueError:
                        height = 3.0
                        
                    osm_buildings.append({
                        "id": f"osm-{way['id']}",
                        "class": tags.get('building', 'yes'),
                        "height": height,
                        "data_date": today_str,
                        "sources": json.dumps([{"dataset": "OpenStreetMap", "update_time": today_str}]),
                        "geometry_polygon_wkt": shapely.geometry.Polygon(coords).wkt,
                        "geom_obj": shapely.geometry.Polygon(coords)
                    })
            return osm_buildings
    except Exception as e:
        print(f"OSM Query failed: {e}", file=sys.stderr)
        return []

def fetch_overture_live(xmin, ymin, xmax, ymax):
    latest_release = overturemaps.core.get_latest_release()
    data_release_version = f"Overture-{latest_release} (Live Cloud)"
    print(f"Querying Overture Maps live via parallelized STAC index pruning (Release: {data_release_version})...")
    bbox_tuple = (xmin, ymin, xmax, ymax)
    reader = overturemaps.record_batch_reader("building", bbox=bbox_tuple, stac=True)
    table = reader.read_all()
    print(f"Fetched {len(table)} records from Overture.")
    
    if len(table) == 0:
        return pd.DataFrame(), data_release_version
        
    wkb_bytes = table.column("geometry").to_pylist()
    geoms = shapely.from_wkb(wkb_bytes)
    
    ids = table.column("id").to_pylist()
    classes = table.column("class").to_pylist()
    heights = [float(h) if h is not None else 3.0 for h in table.column("height").to_pylist()]
    sources = table.column("sources").to_pylist()
    
    max_update_times = [get_max_update_time(s) for s in sources]
    
    df = pd.DataFrame({
        "id": ids,
        "class": classes,
        "height": heights,
        "data_date": max_update_times,
        "sources": [str(s) for s in sources],
        "geometry_polygon_wkt": [g.wkt for g in geoms],
        "geom_obj": geoms
    })
    
    return df, data_release_version

def deduplicate_datasets(df_overture, df_osm):
    if df_overture.empty:
        return df_osm
    if df_osm.empty:
        return df_overture
        
    df_combined = pd.concat([df_overture, df_osm], ignore_index=True)
    df_combined = df_combined.sort_values(by="data_date", ascending=False)
    
    geoms = df_combined["geom_obj"].tolist()
    keep_indices = []
    
    for i, g1 in enumerate(geoms):
        is_dup = False
        for j in keep_indices:
            g2 = geoms[j]
            if g1.intersects(g2):
                intersection_area = g1.intersection(g2).area
                min_area = min(g1.area, g2.area)
                if min_area > 0 and (intersection_area / min_area) > 0.5:
                    is_dup = True
                    break
        if not is_dup:
            keep_indices.append(i)
            
    return df_combined.iloc[keep_indices].copy()

def fetch_latest_satellite_image(xmin, ymin, xmax, ymax):
    print("Searching Microsoft Planetary Computer for the latest cloud-free 2026 Sentinel-2 scene...")
    stac_url = "https://planetarycomputer.microsoft.com/api/stac/v1/search"
    
    query_params = {
        "collections": ["sentinel-2-l2a"],
        "bbox": [xmin, ymin, xmax, ymax],
        "datetime": "2026-01-01T00:00:00Z/2026-12-31T23:59:59Z",
        "query": {
            "eo:cloud_cover": {"lt": 5}
        },
        "sortby": [
            {"field": "properties.datetime", "direction": "desc"}
        ],
        "limit": 1
    }
    
    req = urllib.request.Request(
        stac_url,
        data=json.dumps(query_params).encode('utf-8'),
        headers={'Content-Type': 'application/json', 'User-Agent': 'BuildingExporter/2.0'}
    )
    
    try:
        with urllib.request.urlopen(req) as response:
            search_results = json.loads(response.read().decode('utf-8'))
            features = search_results.get('features', [])
            if not features:
                print("No cloud-free 2026 scenes found. Checking 2025 imagery...")
                query_params["datetime"] = "2025-01-01T00:00:00Z/2025-12-31T23:59:59Z"
                req2 = urllib.request.Request(
                    stac_url,
                    data=json.dumps(query_params).encode('utf-8'),
                    headers={'Content-Type': 'application/json', 'User-Agent': 'BuildingExporter/2.0'}
                )
                with urllib.request.urlopen(req2) as resp2:
                    search_results = json.loads(resp2.read().decode('utf-8'))
                    features = search_results.get('features', [])
                    
            if not features:
                print("No Sentinel-2 scenes found for this bounding box.")
                return None, None, None
                
            item = features[0]
            item_date = item['properties']['datetime'][:10]
            print(f"Found Sentinel-2 scene from: {item_date}")
            
            red_url = item['assets']['B04']['href']
            nir_url = item['assets']['B08']['href']
            
            def sign_url(url):
                sign_api = f"https://planetarycomputer.microsoft.com/api/sas/v1/sign?url={urllib.parse.quote(url)}"
                req_sign = urllib.request.Request(sign_api, headers={'User-Agent': 'BuildingExporter/2.0'})
                with urllib.request.urlopen(req_sign) as res_sign:
                    return json.loads(res_sign.read().decode('utf-8'))['href']
                    
            print("Signing imagery URLs...")
            signed_red = sign_url(red_url)
            signed_nir = sign_url(nir_url)
            return signed_red, signed_nir, item_date
    except Exception as e:
        print(f"STAC imagery search failed: {e}")
        return None, None, None

### Section 3: Data Extraction & Merging

In [ ]:
shapes_to_rasterize = []
df_final = pd.DataFrame()
data_release_version = "Unknown"

if data_source == "live":
    print("Running in default Live mode: Combining Overture Cloud and OpenStreetMap Live...")
    df_overture, overture_release = fetch_overture_live(xmin, ymin, xmax, ymax)
    osm_list = fetch_osm_live(xmin, ymin, xmax, ymax)
    df_osm = pd.DataFrame(osm_list) if osm_list else pd.DataFrame()
    
    df_final = deduplicate_datasets(df_overture, df_osm)
    data_release_version = f"{overture_release} + OSM-Live-Today"
    
elif data_source == "overture_live":
    df_final, data_release_version = fetch_overture_live(xmin, ymin, xmax, ymax)
    
elif data_source == "osm_live":
    osm_list = fetch_osm_live(xmin, ymin, xmax, ymax)
    data_release_version = "OSM-Live-Today"
    if osm_list:
        df_final = pd.DataFrame(osm_list)

# Apply strict Single Latest Date filtering
if not df_final.empty:
    df_final["data_date"] = df_final["data_date"].apply(lambda d: d[:10] if isinstance(d, str) else None)
    valid_dates = df_final["data_date"].dropna()
    
    if not valid_dates.empty:
        latest_date = valid_dates.max()
        print(f"Latest single date identified in the dataset: {latest_date}")
        print(f"Filtering all buildings to keep ONLY this single date...")
        df_final = df_final[df_final["data_date"] == latest_date].copy()
    else:
        print("No valid date field found in dataset. Cleaning data...")
        df_final = pd.DataFrame()

### Section 4: Satellite-based Ghost Building Verification

In [ ]:
if enable_ghost_cleanup and not df_final.empty:
    signed_red, signed_nir, scene_date = fetch_latest_satellite_image(xmin, ymin, xmax, ymax)
    
    if signed_red and signed_nir:
        print(f"Performing automated spectral analysis on buildings using Sentinel-2 image from {scene_date}...")
        try:
            # Load projection packages
            from pyproj import Transformer
            from shapely.ops import transform
            
            with rasterio.open(signed_red) as red_src, rasterio.open(signed_nir) as nir_src:
                # Reproject bounding box coordinates to match Sentinel-2 UTM Projection
                dst_xmin, dst_ymin, dst_xmax, dst_ymax = transform_bounds(
                    "EPSG:4326", red_src.crs, xmin, ymin, xmax, ymax
                )
                # Calculate crop window
                window = win_from_bounds(dst_xmin, dst_ymin, dst_xmax, dst_ymax, transform=red_src.transform)
                
                # Read Red and NIR bands
                red_band = red_src.read(1, window=window).astype(np.float32)
                nir_band = nir_src.read(1, window=window).astype(np.float32)
                
                # Compute NDVI vegetation index array
                ndvi = (nir_band - red_band) / (nir_band + red_band + 1e-6)
                win_transform = rasterio.windows.transform(window, red_src.transform)
                
                transformer = Transformer.from_crs("EPSG:4326", red_src.crs, always_xy=True)
                
                verified_rows = []
                for _, row in df_final.iterrows():
                    geom = row["geom_obj"]
                    proj_geom = transform(transformer.transform, geom)
                    
                    # Create mask of the building polygon on our crop window
                    mask = geometry_mask([proj_geom], out_shape=ndvi.shape, transform=win_transform, invert=True)
                    building_ndvi_pixels = ndvi[mask]
                    
                    if len(building_ndvi_pixels) > 0:
                        mean_ndvi = np.mean(building_ndvi_pixels)
                        # If mean NDVI is high (> 0.35), the building is overgrown/demolished
                        if mean_ndvi > 0.35:
                            continue
                    verified_rows.append(row)
                
                removed_count = len(df_final) - len(verified_rows)
                if removed_count > 0:
                    print(f"Removed {removed_count} ghost buildings that are vegetated or cleared in the {scene_date} imagery.")
                    df_final = pd.DataFrame(verified_rows).copy()
                else:
                    print("All building footprints verified successfully against 2026 imagery!")
                    
        except Exception as e:
            print(f"Satellite cleanup failed: {e}. Keeping raw building footprints.")
    else:
        print("No latest imagery found. Keeping raw building footprints.")

if not df_final.empty:
    df_final["data_release_version"] = data_release_version
    shapes_to_rasterize = list(zip(df_final["geom_obj"], df_final["height"]))

print(f"Extraction complete. {len(df_final)} buildings in final uniform dataset. Data Date: {df_final['data_date'].iloc[0] if not df_final.empty else 'N/A'}")

### Section 5: CSV & GeoTIFF Export

In [ ]:
if df_final.empty:
    # Save an empty CSV file with columns to respect user's request (doesn't matter if it has buildings or not)
    cols = ["id", "class", "height", "data_date", "sources", "geometry_polygon_wkt", "data_release_version"]
    pd.DataFrame(columns=cols).to_csv(csv_out, index=False)
    print(f"No buildings matched the latest date. Exported empty CSV template: {csv_out}")
    
    # Write an empty raster/placeholder if needed or skip
    if os.path.exists(tif_out):
        os.remove(tif_out)
else:
    # Save CSV
    out_cols = [c for c in df_final.columns if c != "geom_obj" and c != "geometry"]
    df_final[out_cols].to_csv(csv_out, index=False)
    print(f"Successfully saved CSV to: {csv_out} (filtered for single latest date)")
    
    # Save GeoTIFF
    print("Generating GeoTIFF raster...")
    width = int(np.ceil((xmax - xmin) / resolution))
    height = int(np.ceil((ymax - ymin) / resolution))
    
    transform = from_bounds(xmin, ymin, xmax, ymax, width, height)
    
    raster = rasterize(
        shapes_to_rasterize,
        out_shape=(height, width),
        transform=transform,
        fill=0.0,
        dtype="float32"
    )
    
    with rasterio.open(
        tif_out,
        'w',
        driver='GTiff',
        height=height,
        width=width,
        count=1,
        dtype='float32',
        crs='EPSG:4326',
        transform=transform,
        nodata=0.0
    ) as dst:
        dst.write(raster, 1)
    print(f"Successfully saved GeoTIFF to: {tif_out}")
    print("Done! You can load these files directly in QGIS.")